In [ ]:
print("hi")

hi


In [ ]:
import pandas as pd

In [ ]:
import sys
!{sys.executable} -m pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00


In [ ]:
# ============================================================
# 🚀 FAST DistilBERT AG News Classifier (Optimized for Colab T4)
# ============================================================
# ✅ Faster Training
# ✅ Better GPU Utilization
# ✅ Automatic Best Model Saving
# ✅ Download Ready
# ✅ VS Code / Deployment Ready
# ============================================================

# ── STEP 0: Install (Run in first Colab cell) ───────────────
# !pip install -q transformers datasets evaluate accelerate scikit-learn

# ── STEP 1: Imports ─────────────────────────────────────────
import os
import numpy as np
import torch
import evaluate

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    pipeline,
)

from sklearn.metrics import classification_report
from google.colab import files

# ── STEP 2: GPU Optimizations ───────────────────────────────
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

device = "cuda" if torch.cuda.is_available() else "cpu"

print("=" * 60)
print(f"🖥️ Device : {device.upper()}")

if device == "cuda":
    print(f"✅ GPU     : {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU not detected! Use T4 GPU in Colab.")
print("=" * 60)

# ── STEP 3: Load Dataset ────────────────────────────────────
print("\n📦 Loading AG News dataset...")
dataset = load_dataset("ag_news")

LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

id2label = {i: label for i, label in enumerate(LABEL_NAMES)}
label2id = {label: i for i, label in enumerate(LABEL_NAMES)}

# ── STEP 4: FAST Stratified Subset ──────────────────────────
TRAIN_SAMPLES = 20000
EVAL_SAMPLES = 2000

def stratified_subset(split, total_samples):

    per_class = total_samples // len(LABEL_NAMES)

    labels = np.array(split["label"])
    indices = []

    for label_id in range(len(LABEL_NAMES)):
        label_indices = np.where(labels == label_id)[0][:per_class]
        indices.extend(label_indices)

    return split.select(indices).shuffle(seed=42)

print("\n✂️ Creating balanced subsets...")

train_subset = stratified_subset(dataset["train"], TRAIN_SAMPLES)
eval_subset = stratified_subset(dataset["test"], EVAL_SAMPLES)

print(f"✅ Train Size : {len(train_subset):,}")
print(f"✅ Eval Size  : {len(eval_subset):,}")

# ── STEP 5: Tokenizer ───────────────────────────────────────
MODEL_CHECKPOINT = "distilbert-base-uncased"

print(f"\n🪙 Loading tokenizer: {MODEL_CHECKPOINT}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_function(examples):

    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128,
    )

print("\n🔄 Tokenizing dataset...")

tokenized_train = train_subset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

tokenized_eval = eval_subset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

# Rename label column
tokenized_train = tokenized_train.rename_column("label", "labels")
tokenized_eval = tokenized_eval.rename_column("label", "labels")

# Torch format
tokenized_train.set_format("torch")
tokenized_eval.set_format("torch")

# Dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# ── STEP 6: Load Model ──────────────────────────────────────
print(f"\n🤖 Loading model: {MODEL_CHECKPOINT}")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_NAMES),
    id2label=id2label,
    label2id=label2id,
)

# ── STEP 7: Metrics ─────────────────────────────────────────
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_metric.compute(
        predictions=predictions,
        references=labels
    )["accuracy"]

    f1 = f1_metric.compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )["f1"]

    return {
        "accuracy": accuracy,
        "f1_score": f1,
    }

# ── STEP 8: Training Arguments ──────────────────────────────
print("\n⚙️ Setting training arguments...")

training_args = TrainingArguments(

    output_dir="./distilbert_news_classifier",

    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_steps=200,

    per_device_train_batch_size=64,
    per_device_eval_batch_size=64,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="f1_score",
    greater_is_better=True,

    save_total_limit=1,

    fp16=torch.cuda.is_available(),

    dataloader_num_workers=2,

    logging_steps=50,

    report_to="none",

    push_to_hub=False,
)

# ── STEP 9: Trainer ─────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
# ── STEP 10: Train ──────────────────────────────────────────
print("\n🚀 TRAINING STARTED")
print("=" * 60)

trainer.train()

# ── STEP 11: Final Evaluation ───────────────────────────────
print("\n📊 Final Evaluation")

results = trainer.evaluate()

print("\n" + "=" * 60)
print(f"✅ Accuracy : {results['eval_accuracy']:.4f}")
print(f"✅ F1 Score : {results['eval_f1_score']:.4f}")
print("=" * 60)

# ── STEP 12: Classification Report ──────────────────────────
print("\n📄 Classification Report")

predictions = trainer.predict(tokenized_eval)

y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

print(
    classification_report(
        y_true,
        y_pred,
        target_names=LABEL_NAMES
    )
)

# ── STEP 13: Save Best Model ────────────────────────────────
SAVE_PATH = "./best_distilbert_news_model"

print(f"\n💾 Saving model to: {SAVE_PATH}")

trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("✅ Model saved successfully!")

# ── STEP 14: Zip Model Folder ───────────────────────────────
print("\n📦 Creating ZIP file...")

!zip -r best_distilbert_news_model.zip best_distilbert_news_model

print("✅ ZIP file created!")

# ── STEP 15: Auto Download ──────────────────────────────────
print("\n⬇️ Downloading ZIP file...")

files.download("best_distilbert_news_model.zip")

# ── STEP 16: Quick Inference Test ───────────────────────────
print("\n🧪 Testing model...")

classifier = pipeline(
    "text-classification",
    model=SAVE_PATH,
    tokenizer=SAVE_PATH,
    device=0 if torch.cuda.is_available() else -1,
)

test_texts = [

    "NASA launches new Mars mission with advanced rover technology.",

    "Real Madrid wins Champions League final in dramatic fashion.",

    "Apple reports record quarterly earnings beating analyst expectations.",

    "Scientists discover new method to treat Alzheimer's disease.",
]

print("\n📌 Sample Predictions")
print("=" * 60)

for text in test_texts:

    result = classifier(text)[0]

    print(f"\n📰 {text}")
    print(f"➡️ Prediction : {result['label']}")
    print(f"🎯 Confidence : {result['score']:.2%}")

print("\n🎉 EVERYTHING COMPLETED SUCCESSFULLY!")

🖥️ Device : CUDA
✅ GPU     : Tesla T4

📦 Loading AG News dataset...

✂️ Creating balanced subsets...
✅ Train Size : 20,000
✅ Eval Size  : 2,000

🪙 Loading tokenizer: distilbert-base-uncased

🔄 Tokenizing dataset...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]


🤖 Loading model: distilbert-base-uncased


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🧪 Testing model...


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]


📌 Sample Predictions

📰 NASA launches new Mars mission with advanced rover technology.
➡️ Prediction : Sci/Tech
🎯 Confidence : 97.42%

📰 Real Madrid wins Champions League final in dramatic fashion.
➡️ Prediction : Sports
🎯 Confidence : 89.52%

📰 Apple reports record quarterly earnings beating analyst expectations.
➡️ Prediction : Sci/Tech
🎯 Confidence : 82.73%

📰 Scientists discover new method to treat Alzheimer's disease.
➡️ Prediction : Sci/Tech
🎯 Confidence : 83.79%

🎉 EVERYTHING COMPLETED SUCCESSFULLY!
